### SetUp

In [ ]:
from project_modules.project_imports import *
from pathlib import Path

from project_modules.classes import Patient
from project_modules.classes import Clinic

from project_modules.rules import call_a_rule

from project_modules.simulation_tools import create_appointments, plot_line_graph
from project_modules.simulation_tools import establish_attendance, random_patient_sample
from project_modules.simulation_tools import get_margin_errors, check_convergence_mean, confidence_interval, calculate_summary

from openpyxl.utils import get_column_letter

In [ ]:
project_path = Path.cwd()
print(project_path)

In [ ]:
def safe_divide(numerator, denominator):
    return numerator / denominator.replace(0, np.nan)

def confidence_bounds(mean_value, margin):
    return mean_value - margin, mean_value + margin

def run_metric_tests(series_a, series_b, metric_name):
    """
    Runs 4 t-tests:
        greater / two-sided
        welch correction True / False
    Returns dataframe.
    """
    tests = []

    configs = [
        ("greater", True),
        ("greater", False),
        ("two-sided", True),
        ("two-sided", False),
    ]

    for alternative, correction in configs:

        df = ttest(
            series_a,
            series_b,
            correction=correction,
            alternative=alternative,
            confidence=0.95
        )

        tests.append({
            "metric": metric_name,
            "alternative": alternative,
            "welch_correction": correction,
            "t_stat": df["T"].values[0],
            "p_value": df["p_val"].values[0]
        })

    return pd.DataFrame(tests)

def autosize_excel_columns(writer, sheet_name):
    ws = writer.sheets[sheet_name]

    for col_cells in ws.columns:
        max_len = max(
            len(str(cell.value)) if cell.value is not None else 0
            for cell in col_cells
        )
        col_letter = get_column_letter(col_cells[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 2, 12)

### Parameter definition

In [ ]:
# ============================================================
# #                   CLINIC CONFIGURATION                   #
# ============================================================

# Configuration Constants
SEED = 42
SLOT_DURATION_MINS = 20
NUM_SERVERS = 1
OPERATING_HOURS = 10
SIMULATION_DAYS = 7
EXTRA_DEMAND_RATE = 0.35 # The percentage of extra patients to simulate beyond capacity

# Derived calculations
SLOTS_PER_DAY = (60 // SLOT_DURATION_MINS) * OPERATING_HOURS
TOTAL_SLOT_CAPACITY = SLOTS_PER_DAY * NUM_SERVERS * SIMULATION_DAYS
TOTAL_PATIENT_POOL = int(TOTAL_SLOT_CAPACITY * (1 + EXTRA_DEMAND_RATE))

print(f"Slots per day: {SLOTS_PER_DAY}")
print(f"Total slot capacity per day: {TOTAL_SLOT_CAPACITY}")
print(f"Total patient pool for simulation: {TOTAL_PATIENT_POOL}")


In [ ]:
# ============================================================
# #                  OVERBOOKING AND POLICY                  #
# ============================================================

RULE_NAME = "simple_pairing"
OVERBOOKING_LEVEL = 12
DISPLACEMENT_OFFSET = 1

In [ ]:
# ============================================================
# #                  SIMULATION PARAMETERS                   #
# ============================================================

PROTECTED_GROUP_PROPORTION = 0.15
NUM_REPLICAS = 35
NUM_ITERATIONS = 35

THRESHOLDS = [
    (0.1, 0.1),
    (0.12, 0.12),
    (0.15, 0.15),
    (0.17, 0.17),
    (0.2, 0.2),
    (0.22, 0.22),
    (0.25, 0.25),
]

# Print thresholds
print("Thresholds for evaluation:")
for idx, (thr_1, thr_2) in enumerate(THRESHOLDS):
    print(f"Thresholds {idx + 1}: Protected = {thr_1}, Unprotected = {thr_2}")

### Initialize patients and load model probabilities

In [ ]:
X_file_path = project_path / "prediction_model/X.npy"
y_file_path = project_path / "prediction_model/y.npy"
column_file_path =  project_path / "prediction_model/column_names.pkl"

with open(column_file_path, 'rb') as file:
    column_names = pickle.load(file)

X = np.load(X_file_path, allow_pickle=True)
y = np.load(y_file_path, allow_pickle=True)

X = pd.DataFrame(X)
X.columns = column_names

patients_original = [Patient(i, **row) for i, row in enumerate(X.to_dict(orient='records'))]

In [ ]:
patient_proba_path = project_path / "patients_with_proba.pkl"

with open(patient_proba_path, 'rb') as f:
    patients_with_proba = pickle.load(f)

patients = patients_with_proba.copy()
probas = [patient.proba for patient in patients]

ppv_npv_results_file_path = "prediction_model/ppv_npv_results.xlsx"
ppv_npv_df = pd.read_excel(ppv_npv_results_file_path)

In [ ]:
# Splitting patients into cohorts
protected_group = [p for p in patients if p.protected]
general_group = [p for p in patients if not p.protected]

# Calculating requirements based on constants
required_protected = int(TOTAL_PATIENT_POOL * PROTECTED_GROUP_PROPORTION)
required_general = TOTAL_PATIENT_POOL - required_protected

# Validation Checks
is_protected_pool_sufficient = len(protected_group) >= required_protected

print(f"--- Population Summary ---")
print(f"Total Available Patients: {len(patients)}")
print(f"Protected Group Size: {len(protected_group)}")
print(f"General Group Size: {len(general_group)}")
print(f"--- Sampling Requirements ---")
print(f"Target Sample Size: {TOTAL_PATIENT_POOL}")
print(f"Protected Patients Needed: {required_protected}")
print(f"General Patients Needed: {required_general}")
print(f"Sufficient Protected Patients: {is_protected_pool_sufficient}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import ks_2samp

prot_probas = [p.proba for p in patients if p.protected]
nonprot_probas = [p.proba for p in patients if not p.protected]

print(f"Protected proba: mean={np.mean(prot_probas):.4f}, std={np.std(prot_probas):.4f}")
print(f"NonProt proba:   mean={np.mean(nonprot_probas):.4f}, std={np.std(nonprot_probas):.4f}")

# Distribution comparison
stat, pval = ks_2samp(prot_probas, nonprot_probas)
print(f"KS test: statistic={stat:.4f}, p-value={pval:.4g}")

# Side-by-side plots
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# Protected
sns.histplot(
    prot_probas,
    color="blue",
    kde=True,
    stat="density",
    bins=30,
    alpha=0.6,
    ax=axes[0]
)
axes[0].set_title("Protected")
axes[0].set_xlabel("Probability")
axes[0].set_ylabel("Density")
axes[0].grid(True, alpha=0.3)

# Non-Protected
sns.histplot(
    nonprot_probas,
    color="red",
    kde=True,
    stat="density",
    bins=30,
    alpha=0.6,
    ax=axes[1]
)
axes[1].set_title("Non-Protected")
axes[1].set_xlabel("Probability")
axes[1].grid(True, alpha=0.3)

plt.suptitle("Probability Distributions Comparison", fontsize=14)
plt.tight_layout()
plt.show()

### Simulation

In [ ]:
# ============================================================
# #                     Helper functions                     #
# ============================================================

def extract_ppv_npv(df, prot_threshold, nonprot_threshold):

    prot_row = df[df["threshold"] == prot_threshold].iloc[0]
    nonprot_row = df[df["threshold"] == nonprot_threshold].iloc[0]

    return {
        "protected_ppv": prot_row["protected_ppv"],
        "non_protected_ppv": nonprot_row["unprotected_ppv"],
        "protected_npv": prot_row["protected_npv"],
        "non_protected_npv": nonprot_row["unprotected_npv"],
    }


def compute_group_metric(df, numerator_col, denominator_col):

    return safe_divide(
        df[numerator_col],
        df[denominator_col]
    ).dropna()


def compute_ci(series):

    mean, margin = confidence_interval(series)

    return {
        "mean": mean,
        "margin": margin,
        "lower": mean - margin,
        "upper": mean + margin,
    }


def build_single_metric_row(base_info, ci_data, series):

    return {
        **base_info,
        "mean": ci_data["mean"],
        "ci_lower": ci_data["lower"],
        "ci_upper": ci_data["upper"],
        "n_replicas": len(series),
    }


def build_paired_metric_row(
    base_info,
    prot_ci,
    nonprot_ci,
    tests_df,
):

    row = {
        **base_info,

        "prot_mean": prot_ci["mean"],
        "prot_ci_lower": prot_ci["lower"],
        "prot_ci_upper": prot_ci["upper"],

        "nonprot_mean": nonprot_ci["mean"],
        "nonprot_ci_lower": nonprot_ci["lower"],
        "nonprot_ci_upper": nonprot_ci["upper"],
    }

    for _, tr in tests_df.iterrows():

        prefix = f"{tr['metric']}_{tr['alternative']}"

        row[f"{prefix}_t_stat"] = tr["t_stat"]
        row[f"{prefix}_p_value"] = tr["p_value"]
        row[f"{prefix}_welch"] = tr["welch_correction"]

    return row


def count_patients(patients, protected=None, flagged=None):

    total = 0

    for p in patients:

        if not p.assigned:
            continue

        if protected is not None and p.protected != protected:
            continue

        if flagged is not None and p.overbooked_target != flagged:
            continue

        total += 1

    return total

In [ ]:
files_prefix = (
    f"{RULE_NAME}_level-{OVERBOOKING_LEVEL}_sample-{TOTAL_PATIENT_POOL}_"
    f"prot-{PROTECTED_GROUP_PROPORTION:.2f}_DISPLACEMENT-{DISPLACEMENT_OFFSET}"
)

DEBUG_VERBOSE = False
model = None

results_summary_rows = []

metric_rows = {
    "PatientWT": [],
    "FlaggedPatientWT": [],
    "StackedPatientWT": [],
    "RCR_Rate": [],
    "FPOR_Rate": [],
    "DR_Rate": [],
    "FDR_Rate": [],
    "SDR_Rate": [],
    "IdleTime": [],
    "OverTime": [],
    "OverallWT": [],
    "TotalAttended": [],
}

# ============================================================
# Metric configuration
# ============================================================

PAIRED_METRICS = {
    "PatientWT": {
        "prot_num": "prot_total_wt",
        "prot_den": "prot_attend",
        "nonprot_num": "nonprot_total_wt",
        "nonprot_den": "nonprot_attend",
    },

    "FlaggedPatientWT": {
        "prot_num": "flagged_wt_prot",
        "prot_den": "flagged_attend_prot",
        "nonprot_num": "flagged_wt_nonprot",
        "nonprot_den": "flagged_attend_nonprot",
    },

    "StackedPatientWT": {
        "prot_num": "stacked_wt_prot",
        "prot_den": "stacked_attend_prot",
        "nonprot_num": "stacked_wt_nonprot",
        "nonprot_den": "stacked_attend_nonprot",
    },

    "RCR_Rate": {
        "prot_num": "crc_prot",
        "prot_den": "prot_attend",
        "nonprot_num": "crc_nonprot",
        "nonprot_den": "nonprot_attend",
    },

    "FPOR_Rate": {
        "prot_num": "crc_prot",
        "prot_den": "ob_assigned_prot",
        "nonprot_num": "crc_nonprot",
        "nonprot_den": "ob_assigned_nonprot",
    },

    "DR_Rate": {
        "prot_num": "cr_prot",
        "prot_den": "prot_attend",
        "nonprot_num": "cr_nonprot",
        "nonprot_den": "nonprot_attend",
    },

    "FDR_Rate": {
        "prot_num": "cr_flagged_prot",
        "prot_den": "flagged_attend_prot",
        "nonprot_num": "cr_flagged_nonprot",
        "nonprot_den": "flagged_attend_nonprot",
    },

    "SDR_Rate": {
        "prot_num": "cr_stacked_prot",
        "prot_den": "stacked_attend_prot",
        "nonprot_num": "cr_stacked_nonprot",
        "nonprot_den": "stacked_attend_nonprot",
    },
}

SINGLE_METRICS = {
    "IdleTime": {
        "numerator": "idle_time",
        "scale": NUM_ITERATIONS * SIMULATION_DAYS,
    },

    "OverTime": {
        "numerator": "over_time",
        "scale": NUM_ITERATIONS * SIMULATION_DAYS,
    },

    "OverallWT": {
        "numerator": "total_wt",
        "denominator": "total_attended",
    },

    "TotalAttended": {
        "numerator": "total_attended",
        "scale": NUM_ITERATIONS * SIMULATION_DAYS,
    },
}


# ============================================================
# Main loop
# ============================================================

for threshold_iter, thresholds in enumerate(THRESHOLDS):

    clear_output(wait=True)

    threshold_protected, threshold_no_protected = thresholds

    ppv_npv = extract_ppv_npv(
        ppv_npv_df,
        threshold_protected,
        threshold_no_protected
    )

    protected_ppv = ppv_npv["protected_ppv"]
    non_protected_ppv = ppv_npv["non_protected_ppv"]

    protected_npv = ppv_npv["protected_npv"]
    non_protected_npv = ppv_npv["non_protected_npv"]

    with open("patients_with_proba.pkl", "rb") as f:
        patients_og = pickle.load(f)

    all_measures = []
    convergence_values = []

    print(
        f"Simulating PT {threshold_protected} & "
        f"NPT {threshold_no_protected} | "
        f"[P PPV {protected_ppv:.4f}] "
        f"[NP PPV {non_protected_ppv:.4f}] "
        f"[P NPV {protected_npv:.4f}] "
        f"[NP NPV {non_protected_npv:.4f}]"
    )

    # ============================================================
    # MONTE CARLO LOOP
    # ============================================================

    convergence = False
    iterations = 0

    while not convergence and iterations < NUM_REPLICAS:

        start = time.time()

        iterations += 1

        patients = copy.deepcopy(patients_og)

        for p in patients:

            if not hasattr(p, "displaced_once"):
                p.displaced_once = False

        created_appointments = create_appointments(
            NUM_SERVERS,
            SIMULATION_DAYS,
            OPERATING_HOURS,
            SLOT_DURATION_MINS
        )

        sampled_random_patient_list = random_patient_sample(
            patients,
            TOTAL_PATIENT_POOL,
            PROTECTED_GROUP_PROPORTION,
            SIMULATION_DAYS
        )

        appointments_from_rule, num_refused = call_a_rule(
            sampled_random_patient_list,
            created_appointments,
            RULE_NAME,
            model,
            threshold_protected,
            threshold_no_protected,
            overbooking_level=OVERBOOKING_LEVEL,
        )

        for inner_iter in range(NUM_ITERATIONS):

            appointments = copy.deepcopy(appointments_from_rule)

            random_patient_list = copy.deepcopy(
                sampled_random_patient_list
            )

            attendance_random_patient_list = establish_attendance(
                random_patient_list,
                protected_ppv,
                non_protected_ppv,
                protected_npv,
                non_protected_npv,
                threshold_protected,
                threshold_no_protected,
            )

            clinic = Clinic(
                attendance_random_patient_list,
                appointments,
                SLOT_DURATION_MINS,
                displacement_offset=DISPLACEMENT_OFFSET
            )

            clinic.simulation()

            measures = clinic.get_measures()

            measures["idle_time_total"] = float(
                sum(measures["idle_time_server"])
            )

            measures["over_time_total"] = float(
                sum(measures["over_time"])
            )

            all_measures.append(measures)

            del (
                appointments,
                random_patient_list,
                attendance_random_patient_list,
                clinic,
            )

        if iterations >= 20:

            tmp_df = pd.DataFrame(all_measures).copy()

            tmp_df["replica_id"] = (
                np.repeat(
                    np.arange(iterations),
                    NUM_ITERATIONS
                )[:len(tmp_df)]
            )

            replica_records = (
                tmp_df
                .drop(
                    columns=["idle_time_server", "over_time"],
                    errors="ignore"
                )
                .groupby("replica_id")
                .mean(numeric_only=True)
                .to_dict(orient="records")
            )

            convergence, diff = check_convergence_mean(
                replica_records,
                converge_mean=0.02,
                verbose=DEBUG_VERBOSE
            )

            convergence_values.append(diff)

        end = time.time()

        print(
            f"Ran Successfully Replica "
            f"{iterations}/{NUM_REPLICAS} "
            f"in {end-start:.2f}s"
        )

        del created_appointments, patients

    print("* * * Ran Successfully All Replicas!!!")

    measures_df = pd.DataFrame(all_measures)

    measures_df["replica_id"] = (
        np.repeat(
            np.arange(iterations),
            NUM_ITERATIONS
        )[:len(measures_df)]
    )

    # ============================================================
    # Population diagnostics
    # ============================================================

    prot_anchor = count_patients(
        sampled_random_patient_list,
        protected=True,
        flagged=False
    )
    nonprot_anchor = count_patients(
        sampled_random_patient_list,
        protected=False,
        flagged=False
    )
    prot_flagged = count_patients(
        sampled_random_patient_list,
        protected=True,
        flagged=True
    )
    nonprot_flagged = count_patients(
        sampled_random_patient_list,
        protected=False,
        flagged=True
    )
    prot_total = prot_anchor + prot_flagged
    nonprot_total = nonprot_anchor + nonprot_flagged
    prot_flagged_pct = (
        prot_flagged / prot_total
        if prot_total > 0 else 0
    )
    nonprot_flagged_pct = (
        nonprot_flagged / nonprot_total
        if nonprot_total > 0 else 0
    )

    # ============================================================
    # Replica aggregation
    # ============================================================

    replica_grouped = (
        measures_df
        .groupby("replica_id")
        .agg(
            prot_total_wt=("clients_total_waiting_time protected class", "sum"),
            nonprot_total_wt=("clients_total_waiting_time non protected class", "sum"),

            prot_attend=("protected_assistance", "sum"),
            nonprot_attend=("non_protected_assistance", "sum"),

            flagged_wt_prot=("cwt_flagged_protected", "sum"),
            flagged_wt_nonprot=("cwt_flagged_non_protected", "sum"),

            flagged_attend_prot=("flagged_protected_attendees", "sum"),
            flagged_attend_nonprot=("flagged_non_protected_attendees", "sum"),

            stacked_wt_prot=("cwt_stacked_protected", "sum"),
            stacked_wt_nonprot=("cwt_stacked_non_protected", "sum"),

            stacked_attend_prot=("stacked_protected_attendees", "sum"),
            stacked_attend_nonprot=("stacked_non_protected_attendees", "sum"),

            crc_prot=("crc_protected", "sum"),
            crc_nonprot=("crc_non_protected", "sum"),

            ob_assigned_prot=("protected_overbooked_assigned", "sum"),
            ob_assigned_nonprot=("non_protected_overbooked_assigned", "sum"),

            cr_prot=("cr_protected", "sum"),
            cr_nonprot=("cr_non_protected", "sum"),

            cr_flagged_prot=("cr_flagged_protected", "sum"),
            cr_flagged_nonprot=("cr_flagged_non_protected", "sum"),

            cr_stacked_prot=("cr_stacked_protected", "sum"),
            cr_stacked_nonprot=("cr_stacked_non_protected", "sum"),

            total_wt=("total_waiting_time", "sum"),
            total_attended=("total_attended_patients", "sum"),

            idle_time=("idle_time_total", "sum"),
            over_time=("over_time_total", "sum"),

            idle_rule_served=("idle_rule_served", "sum"),
            n_fulfilled_served=("n_fulfilled_served", "sum"),
        )
        .reset_index()
    )

    # ============================================================
    # Base row info
    # ============================================================

    base_info = {
        "protected_threshold": threshold_protected,
        "non_protected_threshold": threshold_no_protected,

        "replicas": iterations,
        "inner_iterations": NUM_ITERATIONS,

        "protected_ppv": protected_ppv,
        "non_protected_ppv": non_protected_ppv,

        "protected_npv": protected_npv,
        "non_protected_npv": non_protected_npv,
    }

    # ============================================================
    # Compute paired metrics automatically
    # ============================================================

    paired_metric_results = {}

    for metric_name, cfg in PAIRED_METRICS.items():

        prot_series = compute_group_metric(
            replica_grouped,
            cfg["prot_num"],
            cfg["prot_den"],
        )

        nonprot_series = compute_group_metric(
            replica_grouped,
            cfg["nonprot_num"],
            cfg["nonprot_den"],
        )

        prot_ci = compute_ci(prot_series)
        nonprot_ci = compute_ci(nonprot_series)

        tests_df = run_metric_tests(
            prot_series,
            nonprot_series,
            metric_name
        )

        metric_rows[metric_name].append(
            build_paired_metric_row(
                base_info,
                prot_ci,
                nonprot_ci,
                tests_df,
            )
        )

        paired_metric_results[metric_name] = {
            "prot_series": prot_series,
            "nonprot_series": nonprot_series,
            "prot_ci": prot_ci,
            "nonprot_ci": nonprot_ci,
        }

    # ============================================================
    # Compute single metrics automatically
    # ============================================================

    single_metric_results = {}

    for metric_name, cfg in SINGLE_METRICS.items():

        if "scale" in cfg:

            series = (
                replica_grouped[cfg["numerator"]] /
                cfg["scale"]
            )

        else:

            series = safe_divide(
                replica_grouped[cfg["numerator"]],
                replica_grouped[cfg["denominator"]],
            ).dropna()

        ci = compute_ci(series)

        metric_rows[metric_name].append(
            build_single_metric_row(
                base_info,
                ci,
                series,
            )
        )

        single_metric_results[metric_name] = {
            "series": series,
            "ci": ci,
        }

    # ============================================================
    # Diagnostics
    # ============================================================

    idle_rule_served_replica = (
        replica_grouped["idle_rule_served"] /
        NUM_ITERATIONS
    )

    n_fulfilled_served_replica = (
        replica_grouped["n_fulfilled_served"] /
        NUM_ITERATIONS
    )

    total_displacement_replica = (
        replica_grouped["idle_rule_served"] +
        replica_grouped["n_fulfilled_served"]
    )

    idle_absorption_rate_replica = safe_divide(
        replica_grouped["idle_rule_served"],
        total_displacement_replica,
    ).dropna()

    idle_served_mean, _ = confidence_interval(
        idle_rule_served_replica
    )

    n_fulfilled_mean, _ = confidence_interval(
        n_fulfilled_served_replica
    )

    idle_absorption_rate, _ = confidence_interval(
        idle_absorption_rate_replica
    )

    # ============================================================
    # Summary row
    # ============================================================

    summary_row = {
        "rule": RULE_NAME,

        **base_info,

        "idle_time": (
            single_metric_results["IdleTime"]["ci"]["mean"]
        ),
        "over_time": (
            single_metric_results["OverTime"]["ci"]["mean"]
        ),
        "overall_patient_wt": (
            single_metric_results["OverallWT"]["ci"]["mean"]
        ),
        "total_attended": (
            single_metric_results["TotalAttended"]["ci"]["mean"]
        ),
        "protected_patient_wt": (
            paired_metric_results["PatientWT"]["prot_ci"]["mean"]
        ),
        "non_protected_patient_wt": (
            paired_metric_results["PatientWT"]["nonprot_ci"]["mean"]
        ),
        "protected_flagged_patient_wt": (
            paired_metric_results["FlaggedPatientWT"]["prot_ci"]["mean"]
        ),
        "non_protected_flagged_patient_wt": (
            paired_metric_results["FlaggedPatientWT"]["nonprot_ci"]["mean"]
        ),
        "protected_stacked_patient_wt": (
            paired_metric_results["StackedPatientWT"]["prot_ci"]["mean"]
        ),
        "non_protected_stacked_patient_wt": (
            paired_metric_results["StackedPatientWT"]["nonprot_ci"]["mean"]
        ),
        "protected_RCR_rate": (
            paired_metric_results["RCR_Rate"]["prot_ci"]["mean"]
        ),
        "non_protected_RCR_rate": (
            paired_metric_results["RCR_Rate"]["nonprot_ci"]["mean"]
        ),
        "protected_FPOR_rate": (
            paired_metric_results["FPOR_Rate"]["prot_ci"]["mean"]
        ),
        "non_protected_FPOR_rate": (
            paired_metric_results["FPOR_Rate"]["nonprot_ci"]["mean"]
        ),
        "protected_DR_rate": (
            paired_metric_results["DR_Rate"]["prot_ci"]["mean"]
        ),
        "non_protected_DR_rate": (
            paired_metric_results["DR_Rate"]["nonprot_ci"]["mean"]
        ),
        "protected_FDR_rate": (
            paired_metric_results["FDR_Rate"]["prot_ci"]["mean"]
        ),
        "non_protected_FDR_rate": (
            paired_metric_results["FDR_Rate"]["nonprot_ci"]["mean"]
        ),
        "protected_SDR_rate": (
            paired_metric_results["SDR_Rate"]["prot_ci"]["mean"]
        ),
        "non_protected_SDR_rate": (
            paired_metric_results["SDR_Rate"]["nonprot_ci"]["mean"]
        ),

        "protected_anchors_mean": float(prot_anchor),
        "protected_flagged_mean": float(prot_flagged),

        "non_protected_anchors_mean": float(nonprot_anchor),
        "non_protected_flagged_mean": float(nonprot_flagged),

        "protected_flagged_pct": prot_flagged_pct,
        "non_protected_flagged_pct": nonprot_flagged_pct,

        "flagged_pct_asymmetry": (
            prot_flagged_pct -
            nonprot_flagged_pct
        ),

        "idle_rule_served_mean": idle_served_mean,
        "n_fulfilled_served_mean": n_fulfilled_mean,
        "idle_absorption_rate": idle_absorption_rate,
    }

    results_summary_rows.append(summary_row)


# ============================================================
# WRITE EXCEL
# ============================================================

results_summary_df = pd.DataFrame(results_summary_rows)

base_dir = Path(
    "simulation_outputs/excel_files/post_thesis_iterations"
)

base_dir.mkdir(parents=True, exist_ok=True)

summary_path = (
    base_dir /
    f"{files_prefix}_{NUM_REPLICAS}_replicas_"
    f"{NUM_ITERATIONS}_iter_summary.xlsx"
)

summary_path.unlink(missing_ok=True)

with pd.ExcelWriter(summary_path, engine="openpyxl") as writer:

    results_summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    autosize_excel_columns(writer, "Summary")

    for metric_name, rows in metric_rows.items():

        df_metric = pd.DataFrame(rows)

        sheet_name = metric_name[:31]

        df_metric.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

        autosize_excel_columns(writer, sheet_name)

print(
    f"[summary] Saved "
    f"{len(results_summary_rows)} rows -> "
    f"{summary_path}"
)